# Longest Winning Streak Problem

## Problem Description
We need to calculate the **longest winning streak** for each player.  
- A winning streak is defined as consecutive matches with result = `'Win'`.  
- The streak is interrupted by either `'Draw'` or `'Lose'`.  
- Return the longest streak for each player in any order.

---

## Schema

### Table: Matches
| Column Name | Type | Description                                      |
|-------------|------|--------------------------------------------------|
| player_id   | INT  | Unique identifier for the player                  |
| match_day   | DATE | Date of the match                                |
| result      | ENUM | Result of the match: `'Win'`, `'Draw'`, `'Lose'` |

**Primary Key:** `(player_id, match_day)`

---

## Sample Data

### Matches
| player_id | match_day  | result |
|-----------|------------|--------|
| 1         | 2022-01-17 | Win    |
| 1         | 2022-01-18 | Win    |
| 1         | 2022-01-25 | Win    |
| 1         | 2022-01-31 | Draw   |
| 1         | 2022-02-08 | Win    |
| 2         | 2022-02-06 | Lose   |
| 2         | 2022-02-08 | Lose   |
| 3         | 2022-03-30 | Win    |

---

## Expected Output
| player_id | longest_streak |
|-----------|----------------|
| 1         | 3              |
| 2         | 0              |
| 3         | 1              |

### Explanation
- **Player 1:** Won 3 consecutive matches (Jan 17 → Jan 25). Longest streak = 3.  
- **Player 2:** Only losses, so longest streak = 0.  
- **Player 3:** Single win, so longest streak = 1.  

---

## PySpark Code: Create DataFrame and Temp View

```python


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from datetime import date

# Schema for Matches
matches_schema = StructType([
    StructField("player_id", IntegerType(), False),
    StructField("match_day", DateType(), False),
    StructField("result", StringType(), False)
])

# Data for Matches
matches_data = [
    (1, date(2022,1,17), "Win"),
    (1, date(2022,1,18), "Win"),
    (1, date(2022,1,25), "Win"),
    (1, date(2022,1,31), "Draw"),
    (1, date(2022,2,8), "Win"),
    (2, date(2022,2,6), "Lose"),
    (2, date(2022,2,8), "Lose"),
    (3, date(2022,3,30), "Win")
]

# Create DataFrame
matches_df = spark.createDataFrame(matches_data, matches_schema)

# Register Temp View
matches_df.createOrReplaceTempView("Matches")

# Quick check
matches_df.show()


In [0]:
%sql
WITH cte(SELECT player_id, match_day, result, (
			row_number() OVER (
				PARTITION BY player_id ORDER BY match_day ASC
				) - row_number() OVER (
				PARTITION BY player_id,
				result ORDER BY match_day ASC
				)
			) AS win_streak FROM Matches),
	cte2 AS (
		SELECT DISTINCT c.player_id,
			count(win_streak) OVER (PARTITION BY c.player_id) AS longest_streak
		FROM cte c
		WHERE c.result = 'Win'
			AND win_streak = 0
		)

SELECT DISTINCT m.player_id,
	coalesce(longest_streak, 0) AS longest_streak
FROM matches m
LEFT JOIN cte2 c
	ON m.player_id = c.player_id


# Documentation: Longest Winning Streak Query

## Step 1: Build the Base CTE

```markdown

```sql
WITH cte AS (
    SELECT player_id,
           match_day,
           result,
           (
               ROW_NUMBER() OVER (
                   PARTITION BY player_id ORDER BY match_day ASC
               )
               -
               ROW_NUMBER() OVER (
                   PARTITION BY player_id, result ORDER BY match_day ASC
               )
           ) AS win_streak
    FROM Matches
)
```
- For each player, we assign two row numbers:
  1. `ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY match_day ASC)` → sequential match order per player.
  2. `ROW_NUMBER() OVER (PARTITION BY player_id, result ORDER BY match_day ASC)` → sequential match order per player **within each result category**.
- Subtracting these two row numbers creates a **group identifier (`win_streak`)** that stays constant across consecutive matches with the same result.
- This trick allows us to group consecutive wins together.

---

## Step 2: Count Consecutive Wins (CTE2)
```sql
cte2 AS (
    SELECT DISTINCT 
           c.player_id,
           COUNT(win_streak) OVER (PARTITION BY c.player_id) AS longest_streak
    FROM cte c
    WHERE c.result = 'Win'
      AND win_streak = 0
)
```
- We filter only rows where `result = 'Win'`.
- The condition `win_streak = 0` ensures we only count the **start of each winning streak group**.
- `COUNT(win_streak) OVER (PARTITION BY c.player_id)` counts the number of matches in each streak for that player.
- The result gives us the **longest streak length** per player.

---

## Step 3: Final Selection
```sql
SELECT DISTINCT 
       m.player_id,
       COALESCE(longest_streak, 0) AS longest_streak
FROM Matches m
LEFT JOIN cte2 c
       ON m.player_id = c.player_id
```
- We join back to the `Matches` table to ensure all players are included.
- `COALESCE(longest_streak, 0)` replaces `NULL` with `0` for players who never won a match.
- `DISTINCT` ensures each player appears only once in the final result.

---

## Final Output
- The query returns each `player_id` along with their **longest winning streak**.
- Players with no wins will have a streak of `0`.

---

## Key Insights
1. **Row number subtraction trick** is used to group consecutive identical results.  
2. Filtering on `result = 'Win'` isolates winning streaks.  
3. `COALESCE` ensures players with no wins are handled gracefully.  
4. The query is efficient because it avoids multiple joins and directly leverages window functions.
```

---

This Markdown explains your query step by step, showing how each CTE contributes to calculating the longest winning streak. Would you like me to also add a **worked example with sample data** so you can visualize how the row number subtraction groups consecutive wins?